# Guía Integral de Martingalas en Tiempo Discreto
**Estadística Bayesiana** | UAEMex Valle de México  
Ximena Quintanar Leal · 2025

---


## 1. Definición

Sea $(\Omega, \mathcal{F}, P)$ un espacio de probabilidad y $\{\mathcal{F}_n\}_{n\geq 0}$ una filtración (sucesión creciente de sigma-álgebras).

Una sucesión de variables aleatorias $\{X_n\}_{n\geq 0}$ es una **martingala** respecto a $\{\mathcal{F}_n\}$ si:

1. $X_n$ es $\mathcal{F}_n$-medible (adaptada)
2. $E[|X_n|] < \infty$ para todo $n$
3. $E[X_{n+1} \mid \mathcal{F}_n] = X_n$ casi seguramente

**Interpretación:** dado el historial hasta el momento $n$, la mejor predicción para $X_{n+1}$ es el valor actual $X_n$ — no hay "tendencia" esperada.

- Si $E[X_{n+1} \mid \mathcal{F}_n] \geq X_n$ → **submartingala**
- Si $E[X_{n+1} \mid \mathcal{F}_n] \leq X_n$ → **supermartingala**


## 2. Ejemplo clásico: Caminata aleatoria simple

Sea $\xi_1, \xi_2, \ldots$ i.i.d. con $P(\xi_i = 1) = P(\xi_i = -1) = \tfrac{1}{2}$.

Definimos:
$$S_n = \sum_{i=1}^{n} \xi_i, \qquad S_0 = 0$$

Entonces $\{S_n\}$ es una martingala respecto a $\mathcal{F}_n = \sigma(\xi_1,\ldots,\xi_n)$, porque:

$$E[S_{n+1} \mid \mathcal{F}_n] = S_n + E[\xi_{n+1}] = S_n + 0 = S_n$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(123)

n_pasos = 500
n_trayectorias = 8

pasos = rng.choice([-1, 1], size=(n_trayectorias, n_pasos))
trayectorias = np.cumsum(pasos, axis=1)
trayectorias = np.hstack([np.zeros((n_trayectorias, 1)), trayectorias])

plt.figure(figsize=(9, 4))
for i in range(n_trayectorias):
    plt.plot(trayectorias[i], alpha=0.7, linewidth=1)

plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.title('Caminata aleatoria simple — trayectorias de una martingala')
plt.xlabel('n (paso)')
plt.ylabel('$S_n$')
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print(f'E[S_n] empírico en n={n_pasos}: {trayectorias[:, -1].mean():.3f}  (teórico: 0)')


## 3. Martingala de la urna de Pólya

Una urna tiene inicialmente $r$ bolas rojas y $b$ bolas azules. En cada paso se extrae una bola al azar, se devuelve a la urna **y se añade una bola más del mismo color**.

Sea $X_n$ la proporción de bolas rojas después de $n$ extracciones. Se puede demostrar que $\{X_n\}$ es una martingala:

$$E[X_{n+1} \mid \mathcal{F}_n] = X_n$$

Esto es relevante en estadística bayesiana porque la urna de Pólya está relacionada con el **proceso de Dirichlet** y los modelos no paramétricos bayesianos.


In [ ]:
def simular_urna_polya(r0, b0, pasos, rng):
    r, b = r0, b0
    proporciones = [r / (r + b)]
    for _ in range(pasos):
        total = r + b
        if rng.random() < r / total:
            r += 1   # salió roja, se agrega otra roja
        else:
            b += 1   # salió azul, se agrega otra azul
        proporciones.append(r / (r + b))
    return np.array(proporciones)

plt.figure(figsize=(9, 4))
for i in range(6):
    trayectoria = simular_urna_polya(r0=2, b0=2, pasos=300, rng=rng)
    plt.plot(trayectoria, alpha=0.75, linewidth=1)

plt.title('Urna de Pólya — proporción de bolas rojas (martingala)')
plt.xlabel('n (extracción)')
plt.ylabel('$X_n$ = proporción de rojas')
plt.ylim(0, 1)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


## 4. Teorema de convergencia de martingalas

Si $\{X_n\}$ es una martingala acotada en $L^1$ (es decir, $\sup_n E[|X_n|] < \infty$), entonces existe una variable aleatoria $X_\infty$ tal que:

$$X_n \xrightarrow{c.s.} X_\infty$$

Esto explica por qué en la simulación de la urna de Pólya, cada trayectoria converge a un valor límite distinto — la "información acumulada" determina el destino de cada trayectoria, pero el valor esperado se mantiene constante.


## 5. Conexión con la Estadística Bayesiana

Las martingalas son fundamentales en estadística bayesiana por varias razones:

- **Consistencia de posteriors**: La secuencia de distribuciones a posteriori $\{P(\theta \mid X_1,\ldots,X_n)\}$ forma, bajo ciertas condiciones, una martingala — esto garantiza que el posterior converge al valor verdadero de $\theta$ conforme $n \to \infty$.

- **Procesos de Dirichlet**: La urna de Pólya es el mecanismo generador detrás del proceso de Dirichlet, usado en modelos bayesianos no paramétricos (clustering, mezclas infinitas).

- **MCMC y tiempos de paro**: La teoría de tiempos de paro (*stopping times*), íntimamente ligada a martingalas, es la base teórica detrás de criterios de convergencia en cadenas de Markov Monte Carlo.


## 6. Tiempo de paro — ejemplo

Un **tiempo de paro** $\tau$ es una variable aleatoria tal que el evento $\{\tau = n\}$ depende solo de $\mathcal{F}_n$ (información disponible hasta el momento $n$).

**Teorema de paro opcional (Optional Stopping Theorem):** Si $\{X_n\}$ es martingala y $\tau$ es un tiempo de paro acotado, entonces:

$$E[X_\tau] = E[X_0]$$


In [ ]:
# Ejemplo: tiempo de paro = primera vez que la caminata llega a +5 o -5
def tiempo_de_paro(rng, limite=5, max_pasos=10000):
    s = 0
    for n in range(1, max_pasos + 1):
        s += rng.choice([-1, 1])
        if abs(s) == limite:
            return n, s
    return max_pasos, s

resultados = [tiempo_de_paro(rng) for _ in range(5000)]
tiempos = np.array([r[0] for r in resultados])
valores_finales = np.array([r[1] for r in resultados])

print(f'E[X_tau] empírico:  {valores_finales.mean():.4f}   (teórico: 0)')
print(f'E[tau] empírico:    {tiempos.mean():.2f} pasos')
print(f'P(termina en +5):   {np.mean(valores_finales == 5):.3f}')
print(f'P(termina en -5):   {np.mean(valores_finales == -5):.3f}')


---
## Referencias
- Williams, D. (1991). *Probability with Martingales*. Cambridge University Press.
- Durrett, R. (2019). *Probability: Theory and Examples*, 5th ed.
- Ghosh, J. K. & Ramamoorthi, R. V. (2003). *Bayesian Nonparametrics*. Springer.
